# Reading an ROI crop with a vision-language model (Florence-2)

**Stage 2 of the pipeline.** The ROI pass finds *where* to look on a dense sheet; a VLM reads *what* is there.

This notebook takes a crop from a real site plan and runs **Florence-2** (Microsoft's open VLM, listed in the Conflation CV role) to:
- caption the region,
- describe sub-regions,
- **OCR the callout text** (`EXISTING AC PAVED PARKING (26 STALLS)`, `PROPERTY LINE`, ...).

This is a *prototype of the approach*, not a solved detector. Florence-2 is used pretrained; the production step is **LoRA fine-tuning it on labeled architectural crops** so it answers domain questions (setback vs parcel line, is-this-hatch-parking). Set **Runtime → T4 GPU**, then Run all.

In [ ]:
!pip install -q transformers==4.49.0 timm einops
!pip install -q --upgrade "numpy>=2.0,<2.3"
print("installed — if the next cell errors on numpy, do Runtime > Restart session, then Run all")

> One-time: after the install cell, **Runtime → Restart session → Run all**. The install shuffles numpy; a single restart fixes it.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

def _no_flash(filename):
    return [i for i in get_imports(filename) if i != "flash_attn"]

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
MID = "microsoft/Florence-2-base"

with patch("transformers.dynamic_module_utils.get_imports", _no_flash):
    model = AutoModelForCausalLM.from_pretrained(MID, trust_remote_code=True, torch_dtype=dtype).to(device).eval()
processor = AutoProcessor.from_pretrained(MID, trust_remote_code=True)
print("Florence-2 loaded on", device)

In [ ]:
def florence(task, image, text=""):
    prompt = task + text
    inp = processor(text=prompt, images=image, return_tensors="pt")
    input_ids = inp["input_ids"].to(device)
    pixel_values = inp["pixel_values"].to(device, dtype)
    with torch.no_grad():
        gen = model.generate(input_ids=input_ids, pixel_values=pixel_values,
                             max_new_tokens=1024, num_beams=3, do_sample=False)
    txt = processor.batch_decode(gen, skip_special_tokens=False)[0]
    return processor.post_process_generation(txt, task=task, image_size=(image.width, image.height))

## Upload the site plan

Run this, then pick the drawing file (the Corcoran Ave sheet, or any architectural page).

In [ ]:
from google.colab import files
import io
from PIL import Image

up = files.upload()
name = list(up.keys())[0]
image = Image.open(io.BytesIO(up[name])).convert("RGB")
print("loaded", name, image.size)
image

## Pick an ROI crop

We crop the top-left quadrant here (parking + callouts). Change the fractions to inspect any region a real ROI stage would hand off.

In [ ]:
w, h = image.size
crop = image.crop((0, 0, int(w * 0.5), int(h * 0.55)))
print("crop size", crop.size)
crop

## 1 · What does the VLM see? (caption)

In [ ]:
cap = florence("<MORE_DETAILED_CAPTION>", crop)
print(cap["<MORE_DETAILED_CAPTION>"])

## 2 · Read the callout text (OCR with regions)

The text on the sheet labels the geometry — this is the highest-signal channel for reading a plan.

In [ ]:
ocr = florence("<OCR_WITH_REGION>", crop)
res = ocr["<OCR_WITH_REGION>"]
for lab in res["labels"][:25]:
    print("·", lab.replace("</s>", "").strip())

In [ ]:
from PIL import ImageDraw

def draw_quads(img, res):
    v = img.convert("RGB").copy()
    d = ImageDraw.Draw(v)
    for quad, lab in zip(res["quad_boxes"], res["labels"]):
        pts = [(quad[i], quad[i + 1]) for i in range(0, 8, 2)]
        d.polygon(pts, outline=(249, 115, 22), width=2)
    return v

draw_quads(crop, res)

## 3 · Describe sub-regions (dense region caption)

In [ ]:
dense = florence("<DENSE_REGION_CAPTION>", crop)
r = dense["<DENSE_REGION_CAPTION>"]

def draw_boxes(img, r):
    v = img.convert("RGB").copy()
    d = ImageDraw.Draw(v)
    for box, lab in zip(r["bboxes"], r["labels"]):
        d.rectangle(box, outline=(37, 99, 235), width=2)
        d.text((box[0] + 2, box[1] + 2), lab, fill=(37, 99, 235))
    return v

for box, lab in zip(r["bboxes"], r["labels"]):
    print(f"{lab:30s} {[round(x) for x in box]}")
draw_boxes(crop, r)

## Takeaways — and why this aligns with the role

- A **pretrained VLM already reads the text and proposes regions** with zero training — that alone recovers a big chunk of a sheet's meaning, because architectural drawings label themselves.
- What it **can't** do out of the box: answer domain questions like *"which line is the front setback vs the parcel boundary?"* or *"is this hatch parking?"* Those need the drawing's conventions (dash patterns, hatch textures, bearings).
- **The production path (the internship):** curate labeled crops from real architect drawings, **LoRA fine-tune Florence-2 / Qwen-VL** on them, hold out **by document**, and tie each answer to the zoning/building rule. Wire it behind the ROI + tile-and-stitch stage so a VLM only ever reads the crops that matter.

Full pipeline: **tile → ROI (where) → VLM (what) → ground to code.** This notebook is the *what* stage, prototyped with the exact model family the role uses.